In [75]:
import random
import string
from datetime import datetime

import numpy as np

import psycopg2
from psycopg2 import sql
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT

from pymilvus import (
    connections,
    utility,
    FieldSchema, CollectionSchema, DataType,
    Collection,
)

import torch
from transformers import BertModel, BertTokenizer

# PostGreSQL Setup

In [78]:
# Connect to PostgreSQL
conn = psycopg2.connect("dbname=seek user=seek_user password=secret host=localhost")
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

cur = conn.cursor()

# Drop the database (replace 'seek' with your actual database name)
cur.execute("DROP DATABASE seek")

# Commit changes and close
conn.commit()
cur.close()
conn.close()

InsufficientPrivilege: must be owner of database seek


In [76]:
# Connect to your PostgreSQL database
conn = psycopg2.connect("dbname=postgres user=postgres password=secret host=localhost")
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

# Open a cursor to perform database operations
cur = conn.cursor()

# Create a new database for your project
cur.execute("CREATE DATABASE seek")

# Close communication with the database
cur.close()
conn.close()


DuplicateDatabase: database "seek" already exists


In [9]:
# Connect to the PostgreSQL server
conn = psycopg2.connect("dbname=postgres user=postgres password=secret host=localhost")
conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)
cur = conn.cursor()

# Create a new user
# cur.execute("CREATE USER seek_user WITH PASSWORD 'secret'")

# Grant privileges to the user on the database
cur.execute("GRANT ALL PRIVILEGES ON DATABASE seek TO seek_user")

cur.close()
conn.close()

In [74]:
# Connect to your postgres DB
conn = psycopg2.connect("dbname=seek user=seek_user password=secret host=localhost")

# Open a cursor to perform database operations
cur = conn.cursor()

# Create a table
cur.execute("""
    CREATE TABLE documents(
        id PRIMARY KEY,
        timestamp TIMESTAMP,
        speaker_name TEXT,
        document_title TEXT,
        transcribed_text TEXT
    )
""")

# Commit changes and close
conn.commit()
cur.close()
conn.close()


SyntaxError: syntax error at or near "PRIMARY"
LINE 3:         id PRIMARY KEY,
                   ^


# Milvus Setup

In [59]:
fmt = "\n=== {:30} ===\n"
search_latency_fmt = "search latency = {:.4f}s"
num_entities, dim = 3000, 768

In [60]:
print(fmt.format("start connecting to Milvus"))
connections.connect("default", host="localhost", port="19530")

collection_name = "seek_milvus"
has = utility.has_collection(collection_name)
print(f"Does collection hello_milvus exist in Milvus: {has}")


=== start connecting to Milvus     ===

Does collection hello_milvus exist in Milvus: True


In [63]:
if not has:
    # utility.drop_collection(collection_name=collection_name)
    fields = [
        FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
        FieldSchema(name="embeddings", dtype=DataType.FLOAT_VECTOR, dim=dim)
    ]

    schema = CollectionSchema(fields, "SEEK collection")

    print(fmt.format(f"Create collection {collection_name}"))
    seek_milvus = Collection("seek_milvus", schema, consistency_level="Strong")


# Insert sample data into databases

In [50]:
model = BertModel.from_pretrained('bert-base-uncased')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

config.json: 100%|██████████| 570/570 [00:00<00:00, 5.82MB/s]
model.safetensors: 100%|██████████| 440M/440M [00:03<00:00, 114MB/s]  
tokenizer_config.json: 100%|██████████| 48.0/48.0 [00:00<00:00, 384kB/s]
vocab.txt: 100%|██████████| 232k/232k [00:00<00:00, 1.88MB/s]
tokenizer.json: 100%|██████████| 466k/466k [00:00<00:00, 2.58MB/s]


In [64]:
# Function to generate a random vector
def generate_random_vector(dim):
    return np.random.rand(dim).tolist()

# Function to generate a random sentence
def generate_random_sentence(length):
    # List of words to choose from
    words = ['Lorem', 'ipsum', 'dolor', 'sit', 'amet', 'consectetur', 'adipiscing', 'elit']
    
    # Choose `length` random words from the list
    sentence = ' '.join(random.choices(words, k=length))
    
    return sentence

def generate_embedding(transcribed_text):
    inputs = tokenizer(transcribed_text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    embedding = outputs.last_hidden_state[:,0,:].numpy()
    return embedding.tolist()

# Function to generate dummy metadata
def generate_dummy_metadata(doc_id):
    return {
        'timestamp': datetime.now(),  # current time
        'speaker_name': f'Speaker {doc_id}',
        'document_title': f'Document {doc_id}',
        'transcribed_text': generate_random_sentence(10),  # 10-word random sentence
    }

# Generate sample data
num_documents = 10
dimension = 768  # adjust based on your actual vector size
documents = []
vectors = []

for i in range(num_documents):
    vectors.append(generate_random_vector(dimension))
    documents.append(generate_dummy_metadata(i+1))

In [73]:
print(fmt.format("Start inserting entities"))

# Connect to PostgreSQL
conn = psycopg2.connect("dbname=seek user=seek_user password=secret host=localhost")
cur = conn.cursor()

for i in range(num_documents):
    vector = vectors[i]
    document = documents[i]

    embedding = generate_embedding(document['transcribed_text'])

    # Insert vector into Milvus
    insert_result = seek_milvus.insert([embedding])
    print(insert_result)
    vector_id = insert_result.primary_keys
    print(vector_id[0])

    # Insert the metadata into PostgreSQL, associating it with the vector ID
    cur.execute("""
        INSERT INTO documents (id, timestamp, speaker_name, document_title, transcribed_text)
        VALUES (%s, %s, %s, %s, %s)
        """, (vector_id[0], document['timestamp'], document['speaker_name'], document['document_title'], document['transcribed_text']))

# Commit changes and close
conn.commit()
cur.close()
conn.close()


=== Start inserting entities       ===

(insert count: 1, delete count: 0, upsert count: 0, timestamp: 447939391553536001, success count: 1, err count: 0)
447937425601487910


NumericValueOutOfRange: integer out of range
